In [ ]:
## Tested with T4

In [ ]:
!pip install Cython packaging
!pip install git+https://github.com/NVIDIA/NeMo.git@main#egg=nemo_toolkit[asr]


In [1]:
import nemo.collections.asr as nemo_asr

In [ ]:

asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/nemotron-3.5-asr-streaming-0.6b")


In [4]:
import torchaudio
import torch

audio_path = '/content/test_mono.wav'

# Load the audio file
waveform, sample_rate = torchaudio.load(audio_path)

# Ensure the waveform is mono and remove the channel dimension if present
if waveform.ndim > 1 and waveform.shape[0] > 1:
    waveform = waveform[0, :]
elif waveform.ndim > 1 and waveform.shape[0] == 1:
    waveform = waveform.squeeze(0)

# Resample the audio to 16kHz, a common rate for ASR models
new_sample_rate = 16000
if sample_rate != new_sample_rate:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=new_sample_rate)
    waveform = resampler(waveform)
    sample_rate = new_sample_rate


In [7]:
asr_model.set_inference_prompt("lt-LT")                   # language (or "auto" to detect)
asr_model.encoder.set_default_att_context_size([56, 1])   # latency/accuracy dial

[NeMo I 2026-06-07 21:28:02 mixins:965] Inference prompt set to 'lt-LT' (index 31)


[NeMo W 2026-06-07 21:28:02 conformer_encoder:968] att_context_size=[56, 1] is not among the list of the supported look-aheads: [[56, 3], [56, 0], [56, 6], [56, 13]]


In [14]:
# Pass audio as a list of tensors
transcriptions = asr_model.transcribe(audio=[waveform], language="lt", task="transcribe")



Transcribing: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


In [15]:
print(transcriptions[0].text)

Давис горожа салона сеапсиванки с у летувую kelesimą, paliu negamės susikalbėti litujų kalbą ir gali būti, kad dėl to paslaugos kainuo apigiau, kirpėjo ir grožo specialistų associacijos prezidentė sakė pastebinti jo kartį sužinėči dampingoje kainės ir tikingą, kad sektori šio metų nelenguo. <lt-LT>
